# Ground-Truth State Margin — Phase 2 Embedding

**Phụ thuộc:** chạy sau khi notebook chính đã tạo cache biểu diễn tham chiếu (Capability A).

**Câu hỏi:** `state_margin`/`speaker_margin` trước đó được tính bằng `state_salience` suy từ dự đoán của chính mô hình. Notebook này tính lại bằng nhãn proxy gốc (độc lập, từ đặc trưng âm học thô), theo từng trục A/T/S riêng biệt, để loại bỏ yếu tố tự tham chiếu.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import json, math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import Normalizer

DRIVE_ROOT = Path("/content/drive/MyDrive")
OUTPUT_ROOT = DRIVE_ROOT / "motionemo_v10_eval_capability"
CACHE_ROOT = OUTPUT_ROOT / "_cache"
GT_ROOT = OUTPUT_ROOT / "ground_truth_check"
GT_ROOT.mkdir(parents=True, exist_ok=True)

LOCAL_DATA_ROOT = Path("/content/processed_v10_safe")
DRIVE_DATA_ROOT = DRIVE_ROOT / "data/processed_v10_safe"
DATA_ROOT = LOCAL_DATA_ROOT if (LOCAL_DATA_ROOT / "all_metadata.csv").exists() else DRIVE_DATA_ROOT
METADATA_PATH = DATA_ROOT / "all_metadata.csv"
PRECOMPUTED_DIR = DATA_ROOT / "precomputed"

STATE_THRESHOLDS_PATH = DRIVE_ROOT / "motionemo_v10_phase2/eval/state_thresholds.json"

PHASE1_FP = "bb941e3d94"
PHASE2_FP = "811a04d02e"
DEMO_CFG = dict(reference_split="test", max_reference_samples=1000, reference_random_seed=42)

STATE_NAMES = ["activation", "tension", "stability"]

In [3]:
cache_tag = (f"split{DEMO_CFG['reference_split']}_n{DEMO_CFG['max_reference_samples']}"
             f"_seed{DEMO_CFG['reference_random_seed']}_p1{PHASE1_FP}_p2{PHASE2_FP}")
cache_npz = CACHE_ROOT / f"reference_representations_{cache_tag}.npz"
cache_csv = CACHE_ROOT / f"reference_meta_{cache_tag}.csv"

if not cache_npz.exists():
    raise FileNotFoundError(f"Thiếu cache: {cache_npz}\nChạy Capability A ở notebook chính trước.")

data = np.load(cache_npz)
emb_raw = data["raw"].astype(np.float32)
emb_phase1 = data["phase1"].astype(np.float32)
emb_phase2 = data["phase2"].astype(np.float32)
meta = pd.read_csv(cache_csv)

df_meta = pd.read_csv(METADATA_PATH)
df_meta["feature_path"] = df_meta["feature_path"].apply(lambda p: str(PRECOMPUTED_DIR / Path(str(p)).name))
path_by_sample = df_meta.set_index("sample_id")["feature_path"].to_dict()

with open(STATE_THRESHOLDS_PATH, "r", encoding="utf-8") as f:
    STATE_THRESHOLDS = json.load(f)

print(f"n={len(meta)} | speakers={meta['speaker_uid'].nunique()} | datasets={meta['dataset'].nunique()}")

n=1000 | speakers=20 | datasets=3


In [4]:
# Sao chép NGUYÊN VĂN công thức nhãn proxy từ Training_phase2 / Evaluation_Capability
def _to_np(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def compute_vocal_state_scores_from_arrays(f0_raw, energy_raw, zcr_raw, voiced_mask):
    f0 = _to_np(f0_raw).astype(np.float32).reshape(-1)
    energy = _to_np(energy_raw).astype(np.float32).reshape(-1)
    zcr = _to_np(zcr_raw).astype(np.float32).reshape(-1)
    voiced = _to_np(voiced_mask).astype(np.float32).reshape(-1) > 0.5

    T = min(len(f0), len(energy), len(zcr), len(voiced))
    if T <= 2:
        return {"activation": 0.0, "tension": 0.0, "stability": 0.0}

    f0 = f0[:T]
    energy = np.maximum(energy[:T], 0.0)
    zcr = np.maximum(zcr[:T], 0.0)
    voiced = voiced[:T] & np.isfinite(f0) & (f0 > 0)
    eps = 1e-8

    e_mean = float(np.mean(energy)); e_std = float(np.std(energy))
    e_p95 = float(np.percentile(energy, 95)) + eps
    e_cv = e_std / (e_mean + eps)
    pause_ratio = float(np.mean((energy / e_p95) < 0.03))

    z_mean = float(np.mean(zcr)); z_std = float(np.std(zcr))

    if voiced.sum() > 2:
        vf0 = f0[voiced]
        f0_mean = float(np.mean(vf0)) + eps
        f0_range_rel = (float(np.percentile(vf0, 90)) - float(np.percentile(vf0, 10))) / f0_mean
        f0_motion_rel = float(np.std(np.diff(vf0))) / f0_mean
    else:
        f0_range_rel = 0.0
        f0_motion_rel = 0.0

    log_energy = math.log(e_mean + 1e-6)
    activation = 0.45 * log_energy + 0.35 * z_mean + 0.20 * f0_range_rel
    tension = 0.45 * f0_motion_rel + 0.35 * e_cv + 0.20 * z_std
    instability = 0.45 * pause_ratio + 0.35 * f0_motion_rel + 0.20 * e_cv

    return {"activation": float(activation), "tension": float(tension), "stability": float(-instability)}

def assign_state_label(score, thresholds):
    lo, hi = thresholds
    if score <= lo:
        return 0
    if score <= hi:
        return 1
    return 2

In [5]:
rows = []
for sid in meta["sample_id"]:
    obj = torch.load(path_by_sample[sid], map_location="cpu", weights_only=False)
    scores = compute_vocal_state_scores_from_arrays(
        obj["f0_raw"], obj["energy_raw"], obj["zcr_raw"], obj["voiced_mask"]
    )
    row = {"sample_id": sid}
    for s in STATE_NAMES:
        row[f"{s}_label_true"] = assign_state_label(scores[s], STATE_THRESHOLDS[s])
    rows.append(row)

true_df = pd.DataFrame(rows)
meta_true = meta.merge(true_df, on="sample_id")
meta_true.to_csv(GT_ROOT / "reference_true_labels.csv", index=False)

for s in STATE_NAMES:
    print(s, meta_true[f"{s}_label_true"].value_counts().sort_index().to_dict())

activation {0: 314, 1: 337, 2: 349}
tension {0: 437, 1: 297, 2: 266}
stability {0: 258, 1: 292, 2: 450}


In [6]:
def speaker_center(emb, meta):
    emb = np.asarray(emb, dtype=np.float32)
    centered = emb.copy()
    for spk, idx in meta.groupby("speaker_uid").groups.items():
        centered[idx] -= emb[idx].mean(axis=0)
    return centered

emb_phase2_centered = speaker_center(emb_phase2, meta_true)

In [7]:
def compute_margin_by_label(emb, meta, label_col, n_pairs=30000, seed=42):
    rng = np.random.default_rng(seed)
    X = Normalizer(norm="l2").fit_transform(emb)
    n = len(meta)
    ia = rng.integers(0, n, size=n_pairs)
    ib = rng.integers(0, n, size=n_pairs)
    valid = ia != ib
    ia, ib = ia[valid], ib[valid]

    sim = np.sum(X[ia] * X[ib], axis=1)
    same_spk = meta["speaker_uid"].values[ia] == meta["speaker_uid"].values[ib]
    same_st = meta[label_col].values[ia] == meta[label_col].values[ib]

    mean = lambda m: float(np.mean(sim[m])) if m.any() else float("nan")
    out = dict(
        diff_speaker_same_state=mean(~same_spk & same_st),
        same_speaker_diff_state=mean(same_spk & ~same_st),
    )
    out["state_margin"] = out["diff_speaker_same_state"] - out["same_speaker_diff_state"]
    return out

representations = [
    ("Raw acoustic descriptors", emb_raw),
    ("Phase 1 embedding", emb_phase1),
    ("Phase 2 embedding", emb_phase2),
    ("Phase 2 embedding (speaker-centered)", emb_phase2_centered),
]

rows = []
for name, emb in representations:
    for state in STATE_NAMES:
        m = compute_margin_by_label(emb, meta_true, f"{state}_label_true")
        rows.append({"representation": name, "state": state, **m})

margin_true_df = pd.DataFrame(rows)
margin_true_df.to_csv(GT_ROOT / "state_margin_ground_truth.csv", index=False)
print(margin_true_df.to_string(index=False))

                      representation      state  diff_speaker_same_state  same_speaker_diff_state  state_margin
            Raw acoustic descriptors activation                 0.936880                 0.929267      0.007613
            Raw acoustic descriptors    tension                 0.936620                 0.930571      0.006050
            Raw acoustic descriptors  stability                 0.936504                 0.932195      0.004308
                   Phase 1 embedding activation                 0.689150                 0.790223     -0.101074
                   Phase 1 embedding    tension                 0.686823                 0.786396     -0.099572
                   Phase 1 embedding  stability                 0.689587                 0.786592     -0.097005
                   Phase 2 embedding activation                 0.355799                 0.297610      0.058188
                   Phase 2 embedding    tension                 0.444311                 0.220304      0

In [8]:
print(pd.crosstab(meta_true["dataset"], meta_true["activation_label_true"]))
print()
print(pd.crosstab(meta_true["dataset"], meta_true["tension_label_true"]))
print()
print(pd.crosstab(meta_true["dataset"], meta_true["stability_label_true"]))

activation_label_true    0    1    2
dataset                             
crema                  157  194  198
iemocap                 73  142  151
ravdess                 84    1    0

tension_label_true    0    1    2
dataset                          
crema               235  186  128
iemocap             202  109   55
ravdess               0    2   83

stability_label_true    0    1    2
dataset                            
crema                 135  181  233
iemocap                38  111  217
ravdess                85    0    0


In [9]:
summary = margin_true_df.pivot(index="representation", columns="state", values="state_margin")
summary = summary.reindex([r[0] for r in representations])
summary.to_csv(GT_ROOT / "state_margin_ground_truth_summary.csv")
summary

state,activation,stability,tension
representation,,,
Raw acoustic descriptors,0.007613,0.004308,0.006050
Phase 1 embedding,-0.101074,-0.097005,-0.099572
Phase 2 embedding,0.058188,0.207385,0.224006
Phase 2 embedding (speaker-centered),0.199537,0.296471,0.335805


In [10]:
# === Kiểm định robustness: state_margin giới hạn trong-cùng-dataset ===
# Mục đích: loại trừ khả năng margin đo được ở cell trước bị trộn lẫn với
# hiệu ứng "cùng dataset" (đặc biệt do RAVDESS gần như suy biến 1 nhãn/trục).

def compute_margin_within_dataset(emb, meta, label_col, n_pairs=30000, seed=42):
    rng = np.random.default_rng(seed)
    X = Normalizer(norm="l2").fit_transform(emb)
    n = len(meta)
    ia = rng.integers(0, n, size=n_pairs)
    ib = rng.integers(0, n, size=n_pairs)

    same_ds = meta["dataset"].values[ia] == meta["dataset"].values[ib]
    valid = (ia != ib) & same_ds
    ia, ib = ia[valid], ib[valid]

    sim = np.sum(X[ia] * X[ib], axis=1)
    same_spk = meta["speaker_uid"].values[ia] == meta["speaker_uid"].values[ib]
    same_st = meta[label_col].values[ia] == meta[label_col].values[ib]

    mean = lambda m: float(np.mean(sim[m])) if m.any() else float("nan")
    n_diff_spk_same_st = int((~same_spk & same_st).sum())
    n_same_spk_diff_st = int((same_spk & ~same_st).sum())

    out = dict(
        n_pairs_used=int(valid.sum()),
        n_diff_spk_same_st=n_diff_spk_same_st,
        n_same_spk_diff_st=n_same_spk_diff_st,
        diff_speaker_same_state=mean(~same_spk & same_st),
        same_speaker_diff_state=mean(same_spk & ~same_st),
    )
    out["state_margin"] = out["diff_speaker_same_state"] - out["same_speaker_diff_state"]
    return out


# 1) Toàn tập nhưng giới hạn cặp trong-cùng-dataset (loại bỏ nhiễu cross-dataset)
rows_within = []
for name, emb in representations:
    for state in STATE_NAMES:
        m = compute_margin_within_dataset(emb, meta_true, f"{state}_label_true")
        rows_within.append({"representation": name, "state": state, **m})

margin_within_df = pd.DataFrame(rows_within)
margin_within_df.to_csv(GT_ROOT / "state_margin_within_dataset.csv", index=False)

print("=== State margin, GIỚI HẠN trong-cùng-dataset (toàn tập) ===")
print(margin_within_df[["representation", "state", "n_pairs_used",
                          "n_diff_spk_same_st", "n_same_spk_diff_st",
                          "state_margin"]].to_string(index=False))

# 2) Tách riêng theo từng dataset — quan trọng nhất vì RAVDESS suy biến nhãn
#    nên cần xem CREMA-D và IEMOCAP (2 dataset có phân bố nhãn cân bằng hơn)
#    có tự đứng vững một mình không, không phụ thuộc RAVDESS.
print("\n=== State margin, TÁCH RIÊNG từng dataset ===")
for ds_name in sorted(meta_true["dataset"].unique()):
    sub_mask = (meta_true["dataset"] == ds_name).values
    idx = np.where(sub_mask)[0]
    if len(idx) < 30:
        print(f"[{ds_name}] quá ít mẫu ({len(idx)}), bỏ qua")
        continue

    sub_meta = meta_true.iloc[idx].reset_index(drop=True)
    rows_ds = []
    for name, emb in representations:
        sub_emb = emb[idx]
        for state in STATE_NAMES:
            n_classes = sub_meta[f"{state}_label_true"].nunique()
            if n_classes < 2:
                rows_ds.append({"representation": name, "state": state,
                                 "state_margin": float("nan"),
                                 "note": f"chỉ {n_classes} lớp nhãn, không tính được"})
                continue
            m = compute_margin_by_label(sub_emb, sub_meta, f"{state}_label_true")
            rows_ds.append({"representation": name, "state": state, **m})

    ds_df = pd.DataFrame(rows_ds)
    ds_df.to_csv(GT_ROOT / f"state_margin_within_{ds_name}.csv", index=False)
    print(f"\n--- Dataset: {ds_name} (n={len(idx)}) ---")
    print(ds_df.pivot_table(index="representation", columns="state",
                              values="state_margin",
                              aggfunc="first").reindex([r[0] for r in representations]))

=== State margin, GIỚI HẠN trong-cùng-dataset (toàn tập) ===
                      representation      state  n_pairs_used  n_diff_spk_same_st  n_same_spk_diff_st  state_margin
            Raw acoustic descriptors activation         13354                3699                1713      0.022191
            Raw acoustic descriptors    tension         13354                3933                1563      0.018251
            Raw acoustic descriptors  stability         13354                3931                1505      0.017338
                   Phase 1 embedding activation         13354                3699                1713      0.002220
                   Phase 1 embedding    tension         13354                3933                1563     -0.003527
                   Phase 1 embedding  stability         13354                3931                1505      0.004987
                   Phase 2 embedding activation         13354                3699                1713      0.114864
           